# Setup Medical Billing Knowledge Base

This notebook downloads medical billing codes and builds the vector embedding knowledge base.

## Steps:
1. Verify Docker services are running
2. Download ICD-10-CM codes from CMS
3. Download HCPCS Level II codes from CMS
4. Build embeddings using BioClinical ModernBERT
5. Populate pgvector database
6. Test semantic search

In [ ]:
import sys
import os

# Add project root to path
sys.path.insert(0, '/workspace')

import pandas as pd
from src.data.download_codes import CodeDownloader
from src.data.build_embeddings import build_knowledge_base

## 1. Environment Check

Verify that Docker services are running and database is accessible.

In [ ]:
import psycopg2
from config.settings import DATABASE_URL

try:
    conn = psycopg2.connect(DATABASE_URL)
    with conn.cursor() as cur:
        cur.execute("SELECT version();")
        version = cur.fetchone()[0]
        print("✓ Database connection successful")
        print(f"  PostgreSQL version: {version}")
    conn.close()
except Exception as e:
    print("✗ Database connection failed")
    print(f"  Error: {e}")
    print("\nMake sure Docker is running: docker-compose up -d")

## 2. Download ICD-10-CM Codes

Download diagnosis codes from CMS (public domain).

In [ ]:
downloader = CodeDownloader()

print("Downloading ICD-10-CM codes...")
icd10_df = downloader.download_icd10_codes()

print(f"\n✓ Downloaded {len(icd10_df)} ICD-10 codes")
print(f"  Billable codes: {icd10_df['is_billable'].sum()}")

# Show sample
print("\nSample ICD-10 codes:")
icd10_df.head(10)

## 3. Download HCPCS Level II Codes

Download procedure codes from CMS (public domain, CPT alternative).

In [ ]:
print("Downloading HCPCS codes...")
hcpcs_df = downloader.download_hcpcs_codes()

print(f"\n✓ Downloaded {len(hcpcs_df)} HCPCS codes")
print(f"  Categories: {hcpcs_df['category'].nunique()}")

# Show category distribution
print("\nCategory distribution:")
print(hcpcs_df['category'].value_counts())

# Show sample
print("\nSample HCPCS codes:")
hcpcs_df.head(10)

## 4. Build Embeddings and Populate Database

This step:
1. Loads BioClinical ModernBERT model
2. Generates embeddings for all code descriptions
3. Creates pgvector schema
4. Inserts embeddings with HNSW index

**Note:** This may take 10-20 minutes on first run.

In [ ]:
# Build knowledge base
build_knowledge_base(rebuild=True)

## 5. Verify Database

Check that embeddings were inserted correctly.

In [ ]:
conn = psycopg2.connect(DATABASE_URL)

with conn.cursor() as cur:
    # Count total records
    cur.execute("SELECT COUNT(*) FROM code_embeddings;")
    total = cur.fetchone()[0]
    print(f"✓ Total embeddings: {total}")
    
    # Count by type
    cur.execute("""
        SELECT code_type, COUNT(*) as count
        FROM code_embeddings
        GROUP BY code_type;
    """)
    print("\nBy code type:")
    for code_type, count in cur.fetchall():
        print(f"  {code_type}: {count}")
    
    # Check index
    cur.execute("""
        SELECT indexname
        FROM pg_indexes
        WHERE tablename = 'code_embeddings';
    """)
    print("\nIndexes:")
    for idx in cur.fetchall():
        print(f"  {idx[0]}")

conn.close()

## 6. Test Semantic Search

Test the semantic code search functionality.

In [ ]:
from sentence_transformers import SentenceTransformer
from config.settings import EMBEDDING_MODEL

# Load model
model = SentenceTransformer(EMBEDDING_MODEL)

def search_codes(query, code_type='ICD-10', top_k=5):
    """Search for medical codes using semantic similarity."""
    
    # Embed query
    query_embedding = model.encode(query, convert_to_numpy=True).tolist()
    
    conn = psycopg2.connect(DATABASE_URL)
    
    with conn.cursor() as cur:
        cur.execute("""
            SELECT
                code_id,
                long_description,
                1 - (embedding <=> %s::vector) as similarity
            FROM code_embeddings
            WHERE code_type = %s
            ORDER BY embedding <=> %s::vector
            LIMIT %s;
        """, (query_embedding, code_type, query_embedding, top_k))
        
        results = []
        for code_id, desc, similarity in cur.fetchall():
            results.append({
                'code': code_id,
                'description': desc,
                'similarity': similarity
            })
    
    conn.close()
    return pd.DataFrame(results)

# Test queries
test_queries = [
    "diabetes type 2",
    "high blood pressure",
    "chest pain",
    "heart failure",
    "pneumonia"
]

for query in test_queries:
    print(f"\n{'=' * 60}")
    print(f"Query: '{query}'")
    print(f"{'=' * 60}")
    results = search_codes(query, top_k=3)
    for _, row in results.iterrows():
        print(f"  {row['similarity']:.3f} | {row['code']}: {row['description'][:60]}...")

## Summary

✓ Knowledge base setup complete!

You can now:
- Run NLP extraction pipeline (notebook 5)
- Perform gap analysis (notebook 6)
- Use semantic code search in your applications

**Next steps:**
1. Review sample search results above
2. Proceed to notebook 5 for clinical code extraction
3. Test with your own clinical data (de-identified!)